# Financial Process Optimization Analysis

## Objective
Analyze invoice processing workflow to identify bottlenecks and propose AI-driven optimization solutions.

## Dataset Overview
- **Records:** 500 invoices
- **Time Period:** 12 months
- **Key Variables:** Processing stages, time spent, approval status, error rates

In [ ]:
# Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Set style for professional visualizations
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print("Libraries imported successfully!")

## 1. Load and Explore Data

In [ ]:
# Load dataset
df = pd.read_csv('invoice_processing_data.csv')

# Display basic information
print("Dataset Shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head())
print("\nData Types:")
print(df.dtypes)
print("\nMissing Values:")
print(df.isnull().sum())
print("\nBasic Statistics:")
print(df.describe())

## 2. Data Cleaning & Preprocessing

In [ ]:
# Check for duplicates
print(f"Duplicate records: {df.duplicated().sum()}")

# Handle missing values
df['error_flag'].fillna(0, inplace=True)
df['approval_days'].fillna(df['approval_days'].median(), inplace=True)

# Convert date columns
df['submission_date'] = pd.to_datetime(df['submission_date'])
df['approval_date'] = pd.to_datetime(df['approval_date'])

# Calculate total processing days
df['total_processing_days'] = (df['approval_date'] - df['submission_date']).dt.days

print("\nData cleaned successfully!")
print(f"Total Processing Days - Min: {df['total_processing_days'].min()}, Max: {df['total_processing_days'].max()}, Mean: {df['total_processing_days'].mean():.2f}")

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Key Metrics
print("=" * 50)
print("KEY PERFORMANCE METRICS")
print("=" * 50)

avg_processing_time = df['total_processing_days'].mean()
error_rate = (df['error_flag'].sum() / len(df)) * 100
approval_rate = (df['approval_status'].value_counts().get('Approved', 0) / len(df)) * 100

print(f"Average Processing Time: {avg_processing_time:.2f} days")
print(f"Error Rate: {error_rate:.2f}%")
print(f"First-Time Approval Rate: {approval_rate:.2f}%")

print("\nProcessing Time by Stage:")
print(df.groupby('processing_stage')['stage_duration_days'].describe())

## 4. Bottleneck Identification

In [ ]:
# Bottleneck 1: Processing Time by Stage
stage_analysis = df.groupby('processing_stage').agg({
    'stage_duration_days': ['mean', 'median', 'std', 'max'],
    'invoice_id': 'count'
}).round(2)

stage_analysis.columns = ['Avg Days', 'Median Days', 'Std Dev', 'Max Days', 'Invoice Count']
print("\nBOTTLENECK #1: Processing Time by Stage")
print(stage_analysis.sort_values('Avg Days', ascending=False))

# Identify slowest stage
slowest_stage = df.groupby('processing_stage')['stage_duration_days'].mean().idxmax()
slowest_time = df.groupby('processing_stage')['stage_duration_days'].mean().max()
print(f"\n⚠️  Slowest Stage: {slowest_stage} ({slowest_time:.2f} days avg)")

In [ ]:
# Bottleneck 2: Error Rate Impact
print("\nBOTTLENECK #2: Errors & Rework Impact")

error_analysis = df.groupby('error_flag').agg({
    'total_processing_days': ['mean', 'count'],
    'approval_status': lambda x: (x == 'Approved').sum() / len(x) * 100
}).round(2)

error_analysis.columns = ['Avg Processing Days', 'Count', 'Approval Rate %']
print(error_analysis)

error_impact = df[df['error_flag'] == 1]['total_processing_days'].mean() - df[df['error_flag'] == 0]['total_processing_days'].mean()
print(f"\n⚠️  Error Impact: Invoices with errors take {error_impact:.2f} days LONGER on average")
print(f"    Error Rate: {error_rate:.2f}% of all invoices have errors")

In [ ]:
# Bottleneck 3: Approval Delays
print("\nBOTTLENECK #3: Approval Delays by Department")

dept_analysis = df.groupby('department').agg({
    'approval_days': ['mean', 'median', 'max'],
    'invoice_id': 'count',
    'error_flag': 'sum'
}).round(2)

dept_analysis.columns = ['Avg Approval Days', 'Median Days', 'Max Days', 'Total Invoices', 'Errors']
print(dept_analysis.sort_values('Avg Approval Days', ascending=False))

slowest_dept = df.groupby('department')['approval_days'].mean().idxmax()
slowest_dept_time = df.groupby('department')['approval_days'].mean().max()
print(f"\n⚠️  Slowest Department: {slowest_dept} ({slowest_dept_time:.2f} days avg)")

## 5. Statistical Analysis

In [ ]:
# Correlation Analysis
numeric_cols = df.select_dtypes(include=[np.number]).columns
correlation_matrix = df[numeric_cols].corr()

print("Correlation with Total Processing Days:")
print(correlation_matrix['total_processing_days'].sort_values(ascending=False))

# Statistical Test: Error Impact
error_invoices = df[df['error_flag'] == 1]['total_processing_days']
no_error_invoices = df[df['error_flag'] == 0]['total_processing_days']

t_stat, p_value = stats.ttest_ind(error_invoices, no_error_invoices)
print(f"\nT-Test (Error vs No Error):")
print(f"  T-Statistic: {t_stat:.4f}")
print(f"  P-Value: {p_value:.6f}")
if p_value < 0.05:
    print(f"  ✓ Statistically significant difference (p < 0.05)")

## 6. Visualization: Bottleneck Analysis

In [ ]:
# Visualization 1: Processing Time by Stage
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Chart 1: Average Processing Time by Stage
stage_avg = df.groupby('processing_stage')['stage_duration_days'].mean().sort_values(ascending=False)
colors = ['#d62728' if x > stage_avg.mean() else '#2ca02c' for x in stage_avg.values]
axes[0, 0].barh(stage_avg.index, stage_avg.values, color=colors)
axes[0, 0].axvline(stage_avg.mean(), color='orange', linestyle='--', linewidth=2, label='Average')
axes[0, 0].set_xlabel('Days')
axes[0, 0].set_title('BOTTLENECK #1: Processing Time by Stage', fontweight='bold')
axes[0, 0].legend()

# Chart 2: Error Impact on Processing Time
error_comparison = df.groupby('error_flag')['total_processing_days'].mean()
axes[0, 1].bar(['No Error', 'Error'], error_comparison.values, color=['#2ca02c', '#d62728'], width=0.5)
axes[0, 1].set_ylabel('Days')
axes[0, 1].set_title('BOTTLENECK #2: Error Impact on Processing Time', fontweight='bold')
for i, v in enumerate(error_comparison.values):
    axes[0, 1].text(i, v + 0.5, f'{v:.1f}', ha='center', fontweight='bold')

# Chart 3: Department Performance
dept_avg = df.groupby('department')['approval_days'].mean().sort_values(ascending=False)
colors_dept = ['#d62728' if x > dept_avg.mean() else '#2ca02c' for x in dept_avg.values]
axes[1, 0].barh(dept_avg.index, dept_avg.values, color=colors_dept)
axes[1, 0].axvline(dept_avg.mean(), color='orange', linestyle='--', linewidth=2, label='Average')
axes[1, 0].set_xlabel('Days')
axes[1, 0].set_title('BOTTLENECK #3: Approval Days by Department', fontweight='bold')
axes[1, 0].legend()

# Chart 4: Distribution of Processing Time
axes[1, 1].hist(df['total_processing_days'], bins=30, color='#1f77b4', edgecolor='black', alpha=0.7)
axes[1, 1].axvline(df['total_processing_days'].mean(), color='red', linestyle='--', linewidth=2, label=f"Mean: {df['total_processing_days'].mean():.1f}")
axes[1, 1].axvline(df['total_processing_days'].median(), color='green', linestyle='--', linewidth=2, label=f"Median: {df['total_processing_days'].median():.1f}")
axes[1, 1].set_xlabel('Processing Days')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].set_title('Distribution of Total Processing Time', fontweight='bold')
axes[1, 1].legend()

plt.tight_layout()
plt.savefig('bottleneck_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Visualization saved as 'bottleneck_analysis.png'")

## 7. Key Findings & Recommendations

In [ ]:
print("\n" + "="*70)
print("EXECUTIVE SUMMARY: KEY FINDINGS & RECOMMENDATIONS")
print("="*70)

print(f"""
CURRENT STATE:
  • Average Processing Time: {avg_processing_time:.1f} days
  • Error Rate: {error_rate:.1f}%
  • First-Time Approval Rate: {approval_rate:.1f}%

BOTTLENECK #1 - PROCESSING STAGE DELAYS:
  • Slowest Stage: {slowest_stage} ({slowest_time:.1f} days)
  • Impact: ~{(slowest_time / avg_processing_time * 100):.0f}% of total processing time
  ✓ Recommendation: Implement AI-powered document classification to reduce {slowest_stage} time by 40%

BOTTLENECK #2 - ERROR & REWORK CYCLES:
  • Invoices with Errors: {error_rate:.1f}%
  • Error Impact: +{error_impact:.1f} days processing time
  • Rework Cost: {error_rate:.1f}% of total workflow capacity
  ✓ Recommendation: Deploy ML model for error prediction (reduce errors by 50%, save {error_impact * (error_rate/100) * len(df):.0f} total days)

BOTTLENECK #3 - APPROVAL DELAYS:
  • Slowest Department: {slowest_dept} ({slowest_dept_time:.1f} days)
  • Performance Variance: {dept_avg.max() - dept_avg.min():.1f} days spread
  ✓ Recommendation: Implement intelligent approval routing & automation for routine invoices (target 25% reduction)

OVERALL OPTIMIZATION OPPORTUNITY:
  • Current Total Cycle Time: {avg_processing_time:.1f} days
  • Target (with AI/Automation): {avg_processing_time * 0.65:.1f} days (35% improvement)
  • Estimated Annual Impact: {len(df) * (avg_processing_time - avg_processing_time * 0.65):.0f} days saved
  • Business Value: Faster cash flow, improved vendor relations, reduced manual work
""")

print("="*70)

## 8. Proposed AI Solutions

In [ ]:
print("""
PROPOSED AI & AUTOMATION SOLUTIONS:

1. INTELLIGENT DOCUMENT CLASSIFICATION (NLP/ML)
   - Automatically categorize invoices by type, vendor, amount
   - Route to appropriate processing stage
   - Expected Impact: Reduce Classification stage by 40%

2. PREDICTIVE ERROR DETECTION (Supervised ML)
   - Identify high-risk invoices before manual processing
   - Flag missing data, unusual patterns, discrepancies
   - Expected Impact: Reduce errors by 50%, eliminate rework

3. INTELLIGENT APPROVAL ROUTING (Rules Engine + ML)
   - Auto-approve routine, low-risk invoices
   - Route complex cases to appropriate approver
   - Expected Impact: Reduce approval time by 25%, handle 60% auto-approval

4. PROCESS OPTIMIZATION DASHBOARD (Real-time Monitoring)
   - Real-time visibility into pipeline bottlenecks
   - Alerts for delayed invoices
   - Expected Impact: Enable proactive management, reduce SLA violations

IMPLEMENTATION PRIORITY:
  Phase 1 (Quick Win): Predictive Error Detection → 40% efficiency gain
  Phase 2: Intelligent Approval Routing → Additional 25% improvement
  Phase 3: Full Process Automation → Complete end-to-end optimization
""")

## 9. Data Quality & Methodology

In [ ]:
print(f"""
ANALYSIS METHODOLOGY:
  • Descriptive Statistics: Mean, median, standard deviation analysis
  • Exploratory Data Analysis (EDA): Distribution and pattern identification
  • Statistical Testing: T-tests for significance validation
  • Correlation Analysis: Identify key drivers of processing delays
  • Visualization: Multi-dimensional analysis for stakeholder communication

DATA QUALITY:
  • Total Records: {len(df)}
  • Date Range: {df['submission_date'].min().date()} to {df['submission_date'].max().date()}
  • Missing Values: {df.isnull().sum().sum()} (handled via imputation)
  • Duplicates: {df.duplicated().sum()}
  • Data Completeness: {((len(df) - df.isnull().sum().sum()) / (len(df) * len(df.columns)) * 100):.1f}%
""")

In [ ]:
print("\n✓ Analysis Complete!")
print("\nNext Steps:")
print("  1. Review bottleneck analysis visualizations")
print("  2. Present findings to business stakeholders")
print("  3. Build predictive ML model for error detection")
print("  4. Design AI-powered approval routing system")
print("  5. Create real-time monitoring dashboard")